In [58]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold,RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
import pickle

In [41]:
df_all=pd.read_csv('umap_all_data.csv')
df_umap=pd.read_csv('umap_target_data.csv')

In [42]:
#отделяем признаки от целевой переменной и делаем ее логарифмирование
targets = ['IC50, mM', 'CC50, mM', 'SI']
all_features = [col for col in df_all.columns if col not in targets]
X_all = df_all[all_features].copy()
all_features2 = [col for col in df_umap.columns if col not in targets]
X_umap = df_umap[all_features2].copy()
y_a = df_all['CC50, mM']
y_u = df_umap['CC50, mM']

In [43]:
#делим выборку на обучающую и валидационную
X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(X_all, y_a, test_size=0.2, random_state=42)
X_u_train, X_u_test, y_u_train, y_u_test = train_test_split(X_umap, y_u, test_size=0.2, random_state=42)


In [44]:
#считаем медиану и делаем разметку классов больше/меньше медианы в train и test выборках
median_a_train = y_a_train.median()
y_a_train_cls = (y_a_train > median_a_train).astype(int)
y_a_test_cls = (y_a_test > median_a_train).astype(int)

median_u_train = y_u_train.median()
y_u_train_cls = (y_u_train > median_u_train).astype(int)
y_u_test_cls = (y_u_test > median_u_train).astype(int)

In [45]:
print('Распределение целевой переменной в обучающей выборке')
print(y_a_train_cls.value_counts())
print('Распределение целевой переменной в валидационной выборке')
print(y_a_test_cls.value_counts())

Распределение целевой переменной в обучающей выборке
CC50, mM
1    399
0    399
Name: count, dtype: int64
Распределение целевой переменной в валидационной выборке
CC50, mM
1    108
0     92
Name: count, dtype: int64


In [46]:
scaler_robust = RobustScaler()
X_a_train_r = pd.DataFrame(scaler_robust.fit_transform(X_a_train), columns=X_a_train.columns, index=X_a_train.index)
X_a_test_r = pd.DataFrame(scaler_robust.transform(X_a_test), columns=X_a_test.columns, index=X_a_test.index)

In [47]:
#выбираем нелинейные модели для обучения
#определяем cv для кроссвалидации
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {'Random Forest': {'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {'n_estimators': [200, 300, 350],
                   'max_depth': [10, 20, 30, None],
                   'min_samples_split': [5, 10, 15, 20],
                   'min_samples_leaf': [2, 4, 5]}},
          'Gradient Boosting': {'model': GradientBoostingClassifier(random_state=42),
        'params': {'n_estimators': [200, 300],
                   'learning_rate': [0.01, 0.05, 0.1],
                   'max_depth': [2, 3, 4],
                   'subsample': [0.6, 0.8, 1.0]}},
          'XGBoost': {'model': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
        'params': {'n_estimators': [200, 250],
            'max_depth': [2, 3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.2]}},
          'LightGBM': {'model': lgb.LGBMClassifier(random_state=42, verbose=-1),
        'params': {'n_estimators': [100, 150],
            'num_leaves': [10, 15, 30],
            'learning_rate': [0.05, 0.1, 0.2]}},
          'SVM': {'model': SVC(random_state=42, probability=True),
        'params': {'C': [1, 10, 30, 50],
                   'gamma': ['scale', 'auto', 0.1, 1],
                   'kernel': ['rbf']}},
          'KNN': {'model': KNeighborsClassifier(),
        'params': {'n_neighbors': [3, 5, 7, 11, 15],
                    'weights': ['uniform', 'distance'],
                   'metric': ['euclidean', 'manhattan']}}}


In [48]:
#обучаем модели и сохраняем лучшие результаты
results = []
best_models = {}
print('Модели на всех признаках')
for name, config in models.items():
  grid = GridSearchCV(config['model'], config['params'],
        cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
  grid.fit(X_a_train_r, y_a_train_cls)

  best_model = grid.best_estimator_
  best_models[name] = best_model

  y_pred = best_model.predict(X_a_test_r)
  y_pred_proba = best_model.predict_proba(X_a_test_r)[:, 1]
  results.append({'Model': name,
        'Best Params': str(grid.best_params_),
        'Accuracy': accuracy_score(y_a_test_cls, y_pred),
        'F1-Score': f1_score(y_a_test_cls, y_pred),
        'ROC-AUC': roc_auc_score(y_a_test_cls, y_pred_proba),
        'CV Score': grid.best_score_})
  print(f'\n{name}')
  print(f'Лучшие параметры: {grid.best_params_}')
  print(f'Accuracy: {accuracy_score(y_a_test_cls, y_pred):.3f}')
  print(f'ROC-AUC:  {roc_auc_score(y_a_test_cls, y_pred_proba):.3f}')
  print(f'F1-Score: {f1_score(y_a_test_cls, y_pred):.3f}')

Модели на всех признаках

Random Forest
Лучшие параметры: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 15, 'n_estimators': 300}
Accuracy: 0.750
ROC-AUC:  0.838
F1-Score: 0.762

Gradient Boosting
Лучшие параметры: {'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 300, 'subsample': 0.6}
Accuracy: 0.745
ROC-AUC:  0.833
F1-Score: 0.763

XGBoost
Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200}
Accuracy: 0.745
ROC-AUC:  0.834
F1-Score: 0.751

LightGBM
Лучшие параметры: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 15}
Accuracy: 0.720
ROC-AUC:  0.823
F1-Score: 0.728

SVM
Лучшие параметры: {'C': 10, 'gamma': 0.1, 'kernel': 'rbf'}
Accuracy: 0.685
ROC-AUC:  0.771
F1-Score: 0.704

KNN
Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
Accuracy: 0.740
ROC-AUC:  0.807
F1-Score: 0.748


In [49]:
results = []
best_models = {}
print('Модели на UMAP признаках')
for name, config in models.items():
  grid = GridSearchCV(config['model'], config['params'],
        cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
  grid.fit(X_u_train, y_u_train_cls)

  best_model = grid.best_estimator_
  best_models[name] = best_model

  y_pred = best_model.predict(X_u_test)
  y_pred_proba = best_model.predict_proba(X_u_test)[:, 1]
  results.append({'Model': name,
        'Best Params': str(grid.best_params_),
        'Accuracy': accuracy_score(y_u_test_cls, y_pred),
        'F1-Score': f1_score(y_u_test_cls, y_pred),
        'ROC-AUC': roc_auc_score(y_u_test_cls, y_pred_proba),
        'CV Score': grid.best_score_})
  print(f'\n{name}')
  print(f'Лучшие параметры: {grid.best_params_}')
  print(f'Accuracy: {accuracy_score(y_u_test_cls, y_pred):.3f}')
  print(f'ROC-AUC:  {roc_auc_score(y_u_test_cls, y_pred_proba):.3f}')
  print(f'F1-Score: {f1_score(y_u_test_cls, y_pred):.3f}')

Модели на UMAP признаках

Random Forest
Лучшие параметры: {'max_depth': 30, 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}
Accuracy: 0.680
ROC-AUC:  0.758
F1-Score: 0.692

Gradient Boosting
Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.6}
Accuracy: 0.665
ROC-AUC:  0.747
F1-Score: 0.679

XGBoost
Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 200}
Accuracy: 0.660
ROC-AUC:  0.772
F1-Score: 0.663

LightGBM
Лучшие параметры: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 15}
Accuracy: 0.670
ROC-AUC:  0.746
F1-Score: 0.683

SVM
Лучшие параметры: {'C': 1, 'gamma': 1, 'kernel': 'rbf'}
Accuracy: 0.625
ROC-AUC:  0.733
F1-Score: 0.634

KNN
Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 15, 'weights': 'distance'}
Accuracy: 0.680
ROC-AUC:  0.776
F1-Score: 0.677


В целом модели только на UMAP признаках дают результаты хуже, поэтому продолжим оптимизацию с данными со всеми выбранными признаками

In [50]:
#оптимизируем гиперпараметры лучшей модели с помощью Randomserach
from sklearn.model_selection import RandomizedSearchCV
rf_params_quick = {
    'n_estimators': [1200, 1500, 2000],
    'max_depth': [20, 25],
    'min_samples_split': [5, 10, 15],
    'min_samples_leaf': [1, 2, 3],
    'max_features': [0.3, 0.5],
    'class_weight': ['balanced', 'balanced_subsample']}
random_rf = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
                                rf_params_quick, n_iter=30, cv=3,
                                scoring='balanced_accuracy', random_state=42)
random_rf.fit(X_a_train_r, y_a_train_cls)

RandomizedSearchCV(cv=3,
                   estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
                   n_iter=30,
                   param_distributions={'class_weight': ['balanced',
                                                         'balanced_subsample'],
                                        'max_depth': [20, 25],
                                        'max_features': [0.3, 0.5],
                                        'min_samples_leaf': [1, 2, 3],
                                        'min_samples_split': [5, 10, 15],
                                        'n_estimators': [1200, 1500, 2000]},
                   random_state=42, scoring='balanced_accuracy')

In [56]:
print(f"Best params: {random_rf.best_params_}")
best_rf = random_rf.best_estimator_
y_pred = best_rf.predict(X_a_test_r)
y_pred_proba = best_rf.predict_proba(X_a_test_r)[:, 1]
print(f'Accuracy: {accuracy_score(y_a_test_cls, y_pred):.4f}')
print(f'Test ROC-AUC: {roc_auc_score(y_a_test_cls, y_pred_proba):.4f}')
print(f'F1-Score:  {f1_score(y_a_test_cls, y_pred):.4f}')

Best params: {'n_estimators': 1200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 20, 'class_weight': 'balanced_subsample'}
Accuracy: 0.7500
Test ROC-AUC: 0.8444
F1-Score:  0.7596


In [55]:
#оптимизация GradientBoosting
gb_params_optimized = {
    'n_estimators': [200, 250, 300, 350, 400, 500],
    'learning_rate': [0.005, 0.007, 0.01, 0.015, 0.02, 0.03],
    'max_depth': [2, 3, 4, 5, 6, 7],
    'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'min_samples_split': [2, 3, 5, 7, 10],
    'min_samples_leaf': [1, 2, 3, 4, 5],
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7, 0.9]}
random_gb = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), gb_params_optimized, n_iter=100, cv=cv, scoring='roc_auc', random_state=42,n_jobs=-1,verbose=1)
random_gb.fit(X_a_train_r, y_a_train_cls)
print(f'Best params: {random_gb.best_params_}')
best_r_gb = random_gb.best_estimator_
y_pred_r_gb= best_r_gb.predict(X_a_test_r)
y_pred_proba_r_gb = best_r_gb.predict_proba(X_a_test_r)[:, 1]

print(f'Accuracy:  {accuracy_score(y_a_test_cls, y_pred_r_gb):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_a_test_cls, y_pred_proba_r_gb):.4f}')
print(f'F1-Score:  {f1_score(y_a_test_cls, y_pred_r_gb):.4f}')


Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params: {'subsample': 0.6, 'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 'log2', 'max_depth': 4, 'learning_rate': 0.03}
Accuracy:  0.7300
ROC-AUC:   0.8211
F1-Score:  0.7429


Random forecat дал наилучший результат по всем метрикам. Выбираем эту модельк ак финальную

In [59]:
#сохраняем полученную модель
with open('best_rf_cc50.pkl', 'wb') as f:
    pickle.dump(best_rf, f)
with open('scaler_cc50.pkl', 'wb') as f:
    pickle.dump(scaler_robust, f)
with open('median_cc50.pkl', 'wb') as f:
    pickle.dump(median_a_train, f)
print('Модель и компоненты сохранены в сессионное хранилище')

Модель и компоненты сохранены в сессионное хранилище


In [60]:
#проверяем загрузку
with open('best_rf_cc50.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('scaler_cc50.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
with open('median_cc50.pkl', 'rb') as f:
    loaded_median = pickle.load(f)
print('Модель и компоненты загружены')

Модель и компоненты загружены
